# Etapa 2

In [80]:
import pandas as pd


df_2 = pd.read_excel("data/Datos_etapa 2.xlsx")
df_2.describe()

,labels
count,99.000000
mean,3.010101
std,1.138339
min,1.000000
25%,3.000000
50%,3.000000
75%,4.000000
max,4.000000


In [81]:
df_2.head(10)

,textos,labels
0,La definición de privación está basada en el m...,1
1,La mayoría de los programas de protección soci...,1
2,Esto es dos veces y media mayor que la partici...,1
3,El enfoque aquí está en los aspectos de la pob...,1
4,El factor educativo representa una mayor contr...,1
5,"Se introdujo un nuevo objetivo, relacionado co...",1
6,Este es un aspecto clave de las aspiraciones d...,1
7,Las estimaciones de pobreza y privación multid...,1
8,La mayoría de los indicadores incluidos en la ...,1
9,Esto requiere incluir el cuidado en el análisi...,1


In [82]:
# Limpieza, tokenización y vectorización TF-IDF
from nltk import word_tokenize
import re
from nltk.corpus import stopwords
from nltk import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

spanish_sw = list(stopwords.words("spanish"))
stemmer = SnowballStemmer(language="spanish")
otras_palabras = ["tambien"]
spanish_sw.extend(otras_palabras)
nuevas_stopwords = [stemmer.stem(token) for token in spanish_sw]
spanish_sw.extend(nuevas_stopwords)

def stemmed_tokenizer(text):
    tokens = word_tokenize(text)
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

def clean_text(s):
    s = s.replace("\n", " ").replace("(", " ").replace(")", " ").replace(",", " ").replace(".", " ").replace("?", " ").replace("!", " ").replace("'", " ")
    s = s.lower()
    s = re.sub(r'\d+', ' ', s)
    s = re.sub(r'[^\w]', ' ', s)
    return s

tfidf_word = TfidfVectorizer(
    preprocessor=clean_text,
    tokenizer=stemmed_tokenizer,
    stop_words=spanish_sw,
    max_df=0.9,
    min_df=3,
    max_features=10_000,
    token_pattern=None
)

## Matriz TF-IDF

In [83]:
tfidf_matrix_2 = tfidf_word.fit_transform(df_2["textos"])
tfidf_df_2 = pd.DataFrame(tfidf_matrix_2.toarray(), columns=tfidf_word.get_feature_names_out())

tfidf_df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Columns: 456 entries, abandon to zon
dtypes: float64(456)
memory usage: 352.8 KB


In [84]:
df_2.describe()

,labels
count,99.000000
mean,3.010101
std,1.138339
min,1.000000
25%,3.000000
50%,3.000000
75%,4.000000
max,4.000000


### MODELO 

In [85]:
import joblib
# Guarda la columna original como etiquetas verdaderas
df_2["labels_true"] = df_2["labels"].copy()
# Cargar modelo
modelo = joblib.load('models/modelo.joblib')
# Etiquetar los nuevos datos usando el modelo cargado
predicciones = modelo.predict(df_2["textos"])
df_2["labels"] = predicciones
df_2.head(10)

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.7.1 when using version 1.6.1. This might lead to breaking c

,textos,labels,labels_true
0,La definición de privación está basada en el m...,1,1
1,La mayoría de los programas de protección soci...,3,1
2,Esto es dos veces y media mayor que la partici...,1,1
3,El enfoque aquí está en los aspectos de la pob...,1,1
4,El factor educativo representa una mayor contr...,1,1
5,"Se introdujo un nuevo objetivo, relacionado co...",1,1
6,Este es un aspecto clave de las aspiraciones d...,3,1
7,Las estimaciones de pobreza y privación multid...,1,1
8,La mayoría de los indicadores incluidos en la ...,1,1
9,Esto requiere incluir el cuidado en el análisi...,3,1


### Metricas

In [86]:
from sklearn.metrics import classification_report, accuracy_score


# Compara las etiquetas verdaderas con las predichas
print("Reporte de clasificación (precision, recall, f1-score):")
print(classification_report(df_2["labels_true"], df_2["labels"]))

print("Accuracy:", accuracy_score(df_2["labels_true"], df_2["labels"]))

Reporte de clasificación (precision, recall, f1-score):
              precision    recall  f1-score   support

           1       1.00      0.76      0.86        21
           3       0.88      1.00      0.93        35
           4       0.95      0.95      0.95        43

    accuracy                           0.93        99
   macro avg       0.94      0.91      0.92        99
weighted avg       0.94      0.93      0.93        99

Accuracy: 0.9292929292929293


## Validación de métricas de desempeño sobre el nuevo conjunto de datos

Al aplicar el modelo previamente entrenado sobre el nuevo conjunto de datos (`Datos_etapa 2`), se obtuvieron las siguientes métricas de desempeño: una precisión promedio (macro) de 0.94, un recall promedio de 0.91 y un f1-score macro de 0.92. El accuracy global alcanzado fue de 0.93. Estos resultados indican que el modelo mantiene un alto nivel de rendimiento y generalización, logrando clasificar correctamente la mayoría de las opiniones en las categorías ODS 1, 3 y 4. La consistencia entre las métricas obtenidas en la etapa anterior y las observadas en este nuevo conjunto valida la robustez del modelo y su utilidad para la organización en la automatización de la clasificación de textos.

In [87]:
# Conteo de ejemplos por clase en las etiquetas verdaderas y predichas
print("Conteo por clase en labels_true (original):")
print(df_2["labels_true"].value_counts())

print("\nConteo por clase en labels (predicho):")
print(df_2["labels"].value_counts())

Conteo por clase en labels_true (original):
labels_true
4    43
3    35
1    21
Name: count, dtype: int64

Conteo por clase en labels (predicho):
labels
4    43
3    40
1    16
Name: count, dtype: int64


Como podemos ver aqui la clase minorista en este caso es el ODS 1 ya que en el dataframe original hay 21 apariciones y en las predicciones solo 16 de 99 en todal, por lo que de ahora tendremos que aplicar técnicas de prompting con el fin de generar ejemplos adicionales para la clase minoritaria.

En cuanto al prompting usaremos el asistente IA de Operagx llamado Aria aqui estan las nuevas prompts relacionadas el ODS 1 : fin de la pobreza agregaremos 15 ejemplos nuevos para este ODS 

In [88]:
opiniones_nuevas = [
    "Es fundamental que todos los países trabajen juntos para erradicar la pobreza extrema para 2030. No podemos permitir que nadie se quede atrás.",
    "Crear más oportunidades de empleo es clave para reducir la pobreza. Las personas necesitan trabajos dignos para salir adelante.",
    "La educación es la base para combatir la pobreza. Todos deberían tener acceso a una educación de calidad que les permita mejorar su situación.",
    "Es esencial mejorar el acceso a la protección social, especialmente para las mujeres y los niños, quienes son los más afectados por la pobreza.",
    "La pobreza no solo es un problema económico, sino también social. Debemos abordar la desigualdad que perpetúa la pobreza en nuestras comunidades.",
    "Las iniciativas locales para apoyar a las familias en situación de pobreza son cruciales. Necesitamos más programas que se enfoquen en nuestras comunidades.",
    "La lucha contra la pobreza debe ser sostenible. No podemos simplemente dar ayudas; necesitamos crear sistemas que permitan a las personas ser autosuficientes.",
    "Los países deben colaborar más en la lucha contra la pobreza. La pobreza es un problema global que requiere soluciones globales.",
    "Empoderar a las mujeres es una de las formas más efectivas de reducir la pobreza. Cuando las mujeres prosperan, las comunidades prosperan.",
    "La salud es un factor determinante en la pobreza. Sin acceso a servicios de salud, las personas no pueden trabajar ni cuidar de sus familias.",
    "Invertir en infraestructura básica, como agua potable y saneamiento, es vital para mejorar la calidad de vida y reducir la pobreza.",
    "Es importante crear conciencia sobre la pobreza en nuestra sociedad. Muchas personas no ven la pobreza que existe a su alrededor.",
    "Los programas de microfinanzas pueden ayudar a las personas a iniciar pequeños negocios y salir de la pobreza.",
    "El cambio climático está afectando a los más pobres. Necesitamos abordar este problema para proteger a quienes ya están en situaciones vulnerables.",
    "Los líderes políticos deben comprometerse a implementar políticas efectivas para combatir la pobreza. Las promesas deben convertirse en acciones."
]
# Agregar las nuevas opiniones al DataFrame con el label correspondiente (ODS 1)
nuevos_df = pd.DataFrame({
    "textos": opiniones_nuevas,
    "labels": [1] * len(opiniones_nuevas)
})

# Concatenar al DataFrame original
df_2 = pd.concat([df_2, nuevos_df], ignore_index=True)

df_2.tail(20) 
import pandas as pd

# Dataset original (de la etapa 1)
df_original = pd.read_excel('data/train_data.xlsx')

# Combina los tres conjuntos: original, datos de etapa 2 y opiniones generadas (prompt)
df_total = pd.concat([df_original, df_2], ignore_index=True)


df_total["labels_true"] = df_total["labels"].copy()
# Verifica el resultado
print(df_total.shape)
df_total.head()


(2538, 3)


,textos,labels,labels_true
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4,4
1,Para los niños más pequeños (bebés y niños peq...,4,4
2,"Además, la formación de especialistas en medic...",3,3
3,En los países de la OCDE se tiende a pasar de ...,4,4
4,Este grupo se centró en las personas que padec...,3,3


Para este momento ya tendriamos el conjuntos de datos ampliado con nuevas opiniones de la clase minorista con prompting a Aria, mas lo datos originales y los de esta etapa, por lo que podemos seguir con el reentrenamiento del modelo

# Reentrenamiento

In [89]:
x = df_total["textos"]
y = df_total["labels"]
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42)
import joblib
modelo = joblib.load('models/modelo.joblib')
y_pred = modelo.predict(x_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.7.1 when using version 1.6.1. This might lead to breaking c

              precision    recall  f1-score   support

           1       0.99      0.98      0.98       131
           3       0.99      1.00      0.99       238
           4       1.00      1.00      1.00       266

    accuracy                           0.99       635
   macro avg       0.99      0.99      0.99       635
weighted avg       0.99      0.99      0.99       635

Accuracy: 0.9921259842519685


## Validación del modelo reentrenado

Tras reentrenar el modelo utilizando el conjunto de datos ampliado (original + datos de etapa 2 + ejemplos generados por prompting), se obtuvieron métricas de desempeño significativamente altas: precisión promedio (macro) de 0.99, recall promedio de 0.99 y f1-score macro de 0.99, con una exactitud global (accuracy) de 0.99. Estos resultados demuestran que el modelo logra una excelente capacidad de clasificación en las tres categorías ODS, incluyendo la clase minoritaria ODS 1, que ahora cuenta con mayor representación. El reentrenamiento ha permitido mejorar la robustez y generalización del modelo, asegurando su utilidad para la automatización de la clasificación de textos en la organización.

## Tabla comparativa de métricas de evaluación

A continuación se presenta una tabla comparativa con los valores de precisión, recall, f1-score y accuracy para ambos modelos: el modelo original y el modelo reentrenado tras la aumentación de datos.

| Modelo            | Precisión (macro) | Recall (macro) | F1-score (macro) | Accuracy |
|-------------------|------------------|----------------|------------------|----------|
| Original          | 0.94             | 0.91           | 0.92             | 0.93     |
| Reentrenado       | 0.99             | 0.99           | 0.99             | 0.99     |

**Conclusión:**  
La tabla evidencia mejoras significativas en todas las métricas tras la aumentación de datos y el reentrenamiento, especialmente en la capacidad del modelo para clasificar correctamente la clase minoritaria ODS 1.

In [90]:
# guardar modelo reentrenado
joblib.dump(modelo, 'models/modelo_reentrenado.joblib')

['models/modelo_reentrenado.joblib']

In [91]:
import joblib

# Guardar el vectorizador TF-IDF
joblib.dump(tfidf_word, 'models/tfidf_vectorizer.joblib')

['models/tfidf_vectorizer.joblib']